# 01 - Data Understanding

## Objective
Inspect the raw loan dataset before any transformation: schema, target, missing
values, duplicates, outliers, class balance, and columns that must never be used
as predictors (identifiers or anything only known after the credit decision).

Nothing is modified here -- cleaning happens in notebook 03 / `src/data/preprocessing.py`.

In [1]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd

from src import config
from src.data import preprocessing

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

## Configuration

In [3]:
RAW_FILE = config.RAW_DATA_FILE
RANDOM_STATE = config.RANDOM_STATE
RAW_FILE

WindowsPath('D:/Downloads/ML-Driven-Credit-Risk-Loan-Default-Prediction-System-main/credit-risk-loan-default-ml/data/raw/credit_risk_dataset.csv')

## Data loading

In [5]:
raw = preprocessing.load_raw_data(RAW_FILE)
print(raw.shape)
raw.head()

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
(32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## Target identification

The target is detected from a list of candidate names rather than hard-coded.

In [6]:
TARGET = preprocessing.detect_target(raw)
print("Target column:", TARGET)
raw[TARGET].value_counts(normalize=True).rename("share")

Target column: loan_status


loan_status
0    0.781836
1    0.218164
Name: share, dtype: float64

## Schema, missing values, duplicates

In [7]:
profile = preprocessing.profile_dataframe(raw, TARGET)
print("Rows:", profile["n_rows"], "| Columns:", profile["n_columns"])
print("Duplicate rows:", profile["duplicate_rows"])
print("Missing %:", profile["missing_pct"])
profile["target"]

Rows: 32581 | Columns: 12
Duplicate rows: 165
Missing %: {'person_emp_length': 2.75, 'loan_int_rate': 9.56}


{'name': 'loan_status',
 'counts': {'0': 25473, '1': 7108},
 'positive_rate': 0.2182,
 'imbalance_ratio': 3.58}

In [8]:
raw.dtypes.to_frame("dtype").join(raw.isna().sum().to_frame("missing"))

,dtype,missing
person_age,int64,0
person_income,int64,0
person_home_ownership,object,0
person_emp_length,float64,895
loan_intent,object,0
loan_grade,object,0
loan_amnt,int64,0
loan_int_rate,float64,3116
loan_status,int64,0
loan_percent_income,float64,0


## Categorical levels

In [9]:
for column, levels in profile["categorical_levels"].items():
    print(f"{column}: {levels}")

person_home_ownership: ['MORTGAGE', 'OTHER', 'OWN', 'RENT']
loan_intent: ['DEBTCONSOLIDATION', 'EDUCATION', 'HOMEIMPROVEMENT', 'MEDICAL', 'PERSONAL', 'VENTURE']
loan_grade: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
cb_person_default_on_file: ['N', 'Y']


## Outliers and invalid values

Outliers are *reported*, not deleted: an unusually high income or loan amount is
normally a genuine and informative observation. Values that are physically
impossible (e.g. an age of 144 or 123 years of employment) are a different
matter -- they are data-entry errors and get nulled during cleaning.

In [10]:
preprocessing.outlier_summary(raw)

,column,q1,q3,lower_fence,upper_fence,n_outliers,pct_outliers,min,max
0,person_age,23.00,30.00,12.500,40.500,1494,4.59,20.00,144.00
1,person_income,38500.00,79200.00,-22550.000,140250.000,1484,4.55,4000.00,6000000.00
2,person_emp_length,2.00,7.00,-5.500,14.500,853,2.69,0.00,123.00
3,loan_amnt,5000.00,12200.00,-5800.000,23000.000,1689,5.18,500.00,35000.00
4,loan_int_rate,7.90,13.47,-0.455,21.825,6,0.02,5.42,23.22
5,loan_status,0.00,0.00,0.000,0.000,7108,21.82,0.00,1.00
6,loan_percent_income,0.09,0.23,-0.120,0.440,651,2.00,0.00,0.83
7,cb_person_cred_hist_length,3.00,8.00,-4.500,15.500,1142,3.51,2.00,30.00


In [11]:
for column, (low, high) in config.VALID_RANGES.items():
    if column in raw.columns:
        invalid = raw[(raw[column] < low) | (raw[column] > high)]
        if len(invalid):
            print(f"{column}: {len(invalid)} values outside [{low}, {high}] "
                  f"-> max observed {raw[column].max()}")

person_age: 5 values outside [18, 100] -> max observed 144
person_emp_length: 2 values outside [0, 60] -> max observed 123.0
person_income: 1 values outside [0, 5000000] -> max observed 6000000


## Leakage review

| Column | Known before the credit decision? | Keep as a feature? |
|---|---|---|
| `person_*`, `cb_person_*` | yes -- application and bureau data | yes |
| `loan_amnt`, `loan_intent`, `loan_percent_income` | yes -- requested terms | yes |
| `loan_grade`, `loan_int_rate` | yes -- assigned at underwriting | yes, but they embed the lender's own risk view |
| `loan_status` | no -- the outcome | target |

There is no identifier column in this dataset, and no post-origination field
(payment history, recoveries, delinquency), so target leakage is limited to the
lender-assigned grade/rate. Those are kept because they are genuinely available
at scoring time, and their influence is quantified with SHAP in notebook 06.

In [12]:
print("Identifier-like columns found:", preprocessing.detect_id_columns(raw))

Identifier-like columns found: []


## Results

* Rows / columns and the exact missing-value counts are printed above.
* The target is binary and imbalanced (minority class = defaults).
* Two numeric columns carry impossible values; two carry missing values.
* No identifier column and no post-outcome column are present.

## Interpretation
The dataset is application-level: one row per loan, no time dimension. That rules
out an out-of-time split later on -- a limitation recorded in the README.

## Conclusion
Proceed to EDA (notebook 02) and then to cleaning + feature engineering, where the
invalid values are nulled and imputed **inside the modelling pipeline** so that no
statistic crosses the train/test boundary.